In [14]:
import numpy as np
import pandas as pd
import json
import os
import tarfile
import csv

import gensim, sklearn
import spacy
import sys
sys.path.append(r'C:\Users\Louisa Zhao\anaconda3\Lib\site-packages')
sys.path.append(r'C:\Users\Louisa Zhao\anaconda3\Lib\site-packages\en_core_sci_md')
import scispacy #the package for biomedical and clinical text processing

In [15]:
import lucem_illud #pip install -U git+git://github.com/UChicago-Computational-Content-Analysis/lucem_illud.git

#All these packages need to be installed from pip

#This will be doing most of the work
import networkx as nx

import sklearn #For generating some matrices
import pandas #For DataFrames
import numpy as np #For arrays
import matplotlib.pyplot as plt #For plotting
import seaborn #Makes the plots look nice
import scipy #Some stats
import nltk #a little language code
from IPython.display import Image #for pics

import pickle #if you want to save layouts
import os

%matplotlib inline

In [16]:
nlp = spacy.load(r'C:\Users\Louisa Zhao\anaconda3\Lib\site-packages\en_core_sci_md-0.4.0\en_core_sci_md\en_core_sci_md-0.4.0')

C:\Users\Louisa Zhao\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\util.py:275: UserWarning: [W031] Model 'en_core_sci_md' (0.4.0) requires spaCy v3.0 and is incompatible with the current spaCy version (2.3.5). This may lead to unexpected results or runtime errors. To resolve this, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [17]:
df_cord_meta = pd.read_csv(r"C:\Users\Louisa Zhao\Desktop\CORD\3\metadata.csv",sep=",",header=0)

In [18]:
df_cord_meta

,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,Microsoft Academic Paper ID,WHO #Covidence,has_full_text,full_text_file
0,NaN,Elsevier,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,NaN,4361535.0,els-covid,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,NaN,NaN,False,custom_license
1,NaN,Elsevier,Coronaviruses in Balkan nephritis,10.1016/0002-8703(80)90355-5,NaN,6243850.0,els-covid,NaN,1980-03-31,"Georgescu, Leonida; Diosi, Peter; Buţiu, Ioan;...",American Heart Journal,NaN,NaN,False,custom_license
2,NaN,Elsevier,Cigarette smoking and coronary heart disease: ...,10.1016/0002-8703(80)90356-7,NaN,7355701.0,els-covid,NaN,1980-03-31,"Friedman, Gary D",American Heart Journal,NaN,NaN,False,custom_license
3,aecbc613ebdab36753235197ffb4f35734b5ca63,Elsevier,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,NaN,4579077.0,els-covid,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,NaN,NaN,True,custom_license
4,NaN,Elsevier,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,NaN,4014285.0,els-covid,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,NaN,NaN,False,custom_license
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44215,d4f00f66c732c292fcfc28b19f44daa2fa620901,PMC,Epidemiology and clinical profile of pathogens...,10.1371/journal.pone.0188325,PMC5693464,29149199.0,cc-by,This study aimed to identify a broad spectrum ...,2017 Nov 17,"Brini, Ines; Guerrero, Aida; Hannachi, Naila; ...",PLoS One,NaN,NaN,True,comm_use_subset
44216,ec575d33c0d3b34af7644fcfed64af045a75ab63,Elsevier,Functional Analysis of the Transmembrane Domai...,10.1016/j.jmb.2008.12.029,PMC2750892,19121325.0,els-covid,"Abstract To enter cells, enveloped viruses use...",2009-02-13,"Bissonnette, Mei Lin Z.; Donald, Jason E.; DeG...",Journal of Molecular Biology,NaN,NaN,True,custom_license
44217,7f8715a818bfd325bf4413d3c07003d7ce7b6f7e,PMC,Viral Entry Properties Required for Fitness in...,10.1128/mBio.00898-18,PMC6030562,29970463.0,cc-by,Human parainfluenza viruses cause a large burd...,2018 Jul 3,"Iketani, Sho; Shean, Ryan C.; Ferren, Marion; ...",mBio,NaN,NaN,True,comm_use_subset
44218,07e78e218a159c35e9599e3751a99551a271597b,Elsevier,Arenavirus reverse genetics: New approaches fo...,10.1016/j.virol.2011.01.013,PMC3057228,21324503.0,els-covid,"Abstract Several arenaviruses, chiefly Lassa v...",2011-03-15,"Emonet, Sebastien E.; Urata, Shuzo; de la Torr...",Virology,NaN,NaN,True,custom_license


In [19]:
df_cord_meta['title'] = df_cord_meta['title'].fillna('')
df_cord_meta['abstract'] = df_cord_meta['abstract'].fillna('')
df_cord_meta['title_abstract'] = df_cord_meta.title + " " + df_cord_meta.abstract
df_cord_meta

,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,Microsoft Academic Paper ID,WHO #Covidence,has_full_text,full_text_file,title_abstract
0,NaN,Elsevier,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,NaN,4361535.0,els-covid,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,NaN,NaN,False,custom_license,Intrauterine virus infections and congenital h...
1,NaN,Elsevier,Coronaviruses in Balkan nephritis,10.1016/0002-8703(80)90355-5,NaN,6243850.0,els-covid,,1980-03-31,"Georgescu, Leonida; Diosi, Peter; Buţiu, Ioan;...",American Heart Journal,NaN,NaN,False,custom_license,Coronaviruses in Balkan nephritis
2,NaN,Elsevier,Cigarette smoking and coronary heart disease: ...,10.1016/0002-8703(80)90356-7,NaN,7355701.0,els-covid,,1980-03-31,"Friedman, Gary D",American Heart Journal,NaN,NaN,False,custom_license,Cigarette smoking and coronary heart disease: ...
3,aecbc613ebdab36753235197ffb4f35734b5ca63,Elsevier,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,NaN,4579077.0,els-covid,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,NaN,NaN,True,custom_license,Clinical and immunologic studies in identical ...
4,NaN,Elsevier,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,NaN,4014285.0,els-covid,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,NaN,NaN,False,custom_license,Epidemiology of community-acquired respiratory...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44215,d4f00f66c732c292fcfc28b19f44daa2fa620901,PMC,Epidemiology and clinical profile of pathogens...,10.1371/journal.pone.0188325,PMC5693464,29149199.0,cc-by,This study aimed to identify a broad spectrum ...,2017 Nov 17,"Brini, Ines; Guerrero, Aida; Hannachi, Naila; ...",PLoS One,NaN,NaN,True,comm_use_subset,Epidemiology and clinical profile of pathogens...
44216,ec575d33c0d3b34af7644fcfed64af045a75ab63,Elsevier,Functional Analysis of the Transmembrane Domai...,10.1016/j.jmb.2008.12.029,PMC2750892,19121325.0,els-covid,"Abstract To enter cells, enveloped viruses use...",2009-02-13,"Bissonnette, Mei Lin Z.; Donald, Jason E.; DeG...",Journal of Molecular Biology,NaN,NaN,True,custom_license,Functional Analysis of the Transmembrane Domai...
44217,7f8715a818bfd325bf4413d3c07003d7ce7b6f7e,PMC,Viral Entry Properties Required for Fitness in...,10.1128/mBio.00898-18,PMC6030562,29970463.0,cc-by,Human parainfluenza viruses cause a large burd...,2018 Jul 3,"Iketani, Sho; Shean, Ryan C.; Ferren, Marion; ...",mBio,NaN,NaN,True,comm_use_subset,Viral Entry Properties Required for Fitness in...
44218,07e78e218a159c35e9599e3751a99551a271597b,Elsevier,Arenavirus reverse genetics: New approaches fo...,10.1016/j.virol.2011.01.013,PMC3057228,21324503.0,els-covid,"Abstract Several arenaviruses, chiefly Lassa v...",2011-03-15,"Emonet, Sebastien E.; Urata, Shuzo; de la Torr...",Virology,NaN,NaN,True,custom_license,Arenavirus reverse genetics: New approaches fo...


In [20]:
df_cord_meta['tokenized_sents'] = df_cord_meta['title_abstract'][:100].apply(lambda x: [lucem_illud.word_tokenize(s) for s in lucem_illud.sent_tokenize(x)])

In [22]:
df_cord_meta

,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,Microsoft Academic Paper ID,WHO #Covidence,has_full_text,full_text_file,title_abstract,tokenized_sents
0,NaN,Elsevier,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,NaN,4361535.0,els-covid,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,NaN,NaN,False,custom_license,Intrauterine virus infections and congenital h...,"[[Intrauterine, virus, infections, and, congen..."
1,NaN,Elsevier,Coronaviruses in Balkan nephritis,10.1016/0002-8703(80)90355-5,NaN,6243850.0,els-covid,,1980-03-31,"Georgescu, Leonida; Diosi, Peter; Buţiu, Ioan;...",American Heart Journal,NaN,NaN,False,custom_license,Coronaviruses in Balkan nephritis,"[[Coronaviruses, in, Balkan, nephritis]]"
2,NaN,Elsevier,Cigarette smoking and coronary heart disease: ...,10.1016/0002-8703(80)90356-7,NaN,7355701.0,els-covid,,1980-03-31,"Friedman, Gary D",American Heart Journal,NaN,NaN,False,custom_license,Cigarette smoking and coronary heart disease: ...,"[[Cigarette, smoking, and, coronary, heart, di..."
3,aecbc613ebdab36753235197ffb4f35734b5ca63,Elsevier,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,NaN,4579077.0,els-covid,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,NaN,NaN,True,custom_license,Clinical and immunologic studies in identical ...,"[[Clinical, and, immunologic, studies, in, ide..."
4,NaN,Elsevier,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,NaN,4014285.0,els-covid,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,NaN,NaN,False,custom_license,Epidemiology of community-acquired respiratory...,"[[Epidemiology, of, community, acquired, respi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44215,d4f00f66c732c292fcfc28b19f44daa2fa620901,PMC,Epidemiology and clinical profile of pathogens...,10.1371/journal.pone.0188325,PMC5693464,29149199.0,cc-by,This study aimed to identify a broad spectrum ...,2017 Nov 17,"Brini, Ines; Guerrero, Aida; Hannachi, Naila; ...",PLoS One,NaN,NaN,True,comm_use_subset,Epidemiology and clinical profile of pathogens...,NaN
44216,ec575d33c0d3b34af7644fcfed64af045a75ab63,Elsevier,Functional Analysis of the Transmembrane Domai...,10.1016/j.jmb.2008.12.029,PMC2750892,19121325.0,els-covid,"Abstract To enter cells, enveloped viruses use...",2009-02-13,"Bissonnette, Mei Lin Z.; Donald, Jason E.; DeG...",Journal of Molecular Biology,NaN,NaN,True,custom_license,Functional Analysis of the Transmembrane Domai...,NaN
44217,7f8715a818bfd325bf4413d3c07003d7ce7b6f7e,PMC,Viral Entry Properties Required for Fitness in...,10.1128/mBio.00898-18,PMC6030562,29970463.0,cc-by,Human parainfluenza viruses cause a large burd...,2018 Jul 3,"Iketani, Sho; Shean, Ryan C.; Ferren, Marion; ...",mBio,NaN,NaN,True,comm_use_subset,Viral Entry Properties Required for Fitness in...,NaN
44218,07e78e218a159c35e9599e3751a99551a271597b,Elsevier,Arenavirus reverse genetics: New approaches fo...,10.1016/j.virol.2011.01.013,PMC3057228,21324503.0,els-covid,"Abstract Several arenaviruses, chiefly Lassa v...",2011-03-15,"Emonet, Sebastien E.; Urata, Shuzo; de la Torr...",Virology,NaN,NaN,True,custom_license,Arenavirus reverse genetics: New approaches fo...,NaN


In [21]:
from gensim.parsing.preprocessing import remove_stopwords
import string
all_stopwords = nlp.Defaults.stop_words

In [25]:
%%time
processed_docs = list()
for i in range(len(df_cord_meta['title_abstract'][20000:])):
    text = df_cord_meta['title_abstract'][i+20000]
    doc = nlp(text)
    tokens = [token.text for token in doc]
    tokens_without_sw= [word.lower() for word in tokens if not word in all_stopwords]
    sentence = ''
    for t in tokens_without_sw:
        sentence = sentence + t +" "
        
    df_cord_meta['tokenized_sents'][i+20000] = sentence 
    #processed_docs.append(tokens_without_sw)

<timed exec>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Wall time: 2min 20s


In [26]:
df_cord_meta['tokenized_sents']

0        intrauterine virus infections congenital heart...
1                          coronaviruses balkan nephritis 
2        cigarette smoking coronary heart disease : new...
3        clinical immunologic studies identical twins d...
4        epidemiology community-acquired respiratory tr...
                               ...                        
44215    epidemiology clinical profile pathogens respon...
44216    functional analysis transmembrane domain param...
44217    viral entry properties required fitness humans...
44218    arenavirus reverse genetics : new approaches i...
44219    a new immunosuppressive molecule emodin induce...
Name: tokenized_sents, Length: 44220, dtype: object

In [27]:
def clean_year(year):
    if pd.isna(year):
        return np.nan
    else:
        return int(year[:4])

In [28]:
df_cord_meta['year']=df_cord_meta['publish_time'].apply(clean_year)
df_cord_meta

ValueError: invalid literal for int() with base 10: "['20"

In [29]:
df_csv = df_cord_meta

In [43]:
df_small = df_csv.drop(['title','journal','Microsoft Academic Paper ID','WHO #Covidence','has_full_text','full_text_file','sha','source_x','doi','pmcid','pubmed_id','license','title_abstract','abstract','publish_time','authors'],axis=1)
df_small['abstract'] = df_small['tokenized_sents']
df_save = df_small.drop(['tokenized_sents'],axis=1)
df_save

,abstract
0,intrauterine virus infections congenital heart...
1,coronaviruses balkan nephritis
2,cigarette smoking coronary heart disease : new...
3,clinical immunologic studies identical twins d...
4,epidemiology community-acquired respiratory tr...
...,...
44215,epidemiology clinical profile pathogens respon...
44216,functional analysis transmembrane domain param...
44217,viral entry properties required fitness humans...
44218,arenavirus reverse genetics : new approaches i...


In [48]:
file_name = "C:/Users/Louisa Zhao/Desktop/token.csv"
file_name_small = "C:/Users/Louisa Zhao/Desktop/token small - 3.csv"

In [52]:
df_save

,abstract
0,intrauterine virus infections congenital heart...
1,coronaviruses balkan nephritis
2,cigarette smoking coronary heart disease : new...
3,clinical immunologic studies identical twins d...
4,epidemiology community-acquired respiratory tr...
...,...
44215,epidemiology clinical profile pathogens respon...
44216,functional analysis transmembrane domain param...
44217,viral entry properties required fitness humans...
44218,arenavirus reverse genetics : new approaches i...


In [12]:
df_cord = pd.read_csv(file_name,sep=",",header=0)
df_new = df_cord[:200119]

In [54]:
df_save

,abstract
0,intrauterine virus infections congenital heart...
1,coronaviruses balkan nephritis
2,cigarette smoking coronary heart disease : new...
3,clinical immunologic studies identical twins d...
4,epidemiology community-acquired respiratory tr...
...,...
44215,epidemiology clinical profile pathogens respon...
44216,functional analysis transmembrane domain param...
44217,viral entry properties required fitness humans...
44218,arenavirus reverse genetics : new approaches i...


In [55]:
df_save.to_csv(file_name_small,sep=',', encoding='utf-8',index=False)

In [56]:
pd.read_csv(file_name_small)

,abstract
0,intrauterine virus infections congenital heart...
1,coronaviruses balkan nephritis
2,cigarette smoking coronary heart disease : new...
3,clinical immunologic studies identical twins d...
4,epidemiology community-acquired respiratory tr...
...,...
44140,epidemiology clinical profile pathogens respon...
44141,functional analysis transmembrane domain param...
44142,viral entry properties required fitness humans...
44143,arenavirus reverse genetics : new approaches i...
